# Experiment 41 — Backward-free SparseWalker with EMA targets

Surgical extension of Experiment 39. Same corrected SparseWalker v1.1 recurrence and same local update rules; the only new mechanism is a **slow EMA copy of the item representation** used as the teaching target.

- online item table: forward scoring + plastic updates
- EMA item table: positive/negative teaching vectors, router target context, concept-value target
- no warm start, no optimizer, no `backward()`, no autograd learning
- `last.pt` is saved every epoch for crash recovery


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os,sys,shutil,subprocess,runpy,json,torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='agent/local-contrastive-ema-targets-v2'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments',f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(),'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Run / resume

Default is `EMA=0.995`. Set `RESUME=True` after a runtime crash.


In [ ]:
EMA=0.995
RESUME=False
SCRIPT=f'{REPO}/experiments/run_amazon_local_contrastive_ema.py'
argv=[SCRIPT,'--dataset','beauty','--epochs','70','--batch-size','512','--eval-batch-size','1024','--ema',str(EMA)]
if RESUME: argv.append('--resume')
sys.argv=argv
runpy.run_path(SCRIPT,run_name='__main__')


## Inspect trajectory

The main target is to beat Experiment 39's **0.040516 validation NDCG@10** and ideally SASRec's **0.042968**. `mean_online_target_cosine` tells us how far the fast and slow representations separate.


In [ ]:
import pandas as pd
root=Path(f'/content/drive/MyDrive/sparsewalker_local_contrastive_ema/beauty/seed42/ema{EMA:g}')
hp=root/'history.json'
if hp.exists():
    h=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','mean_contrastive_margin','mean_positive_prob','mean_negative_prob','mean_online_target_cosine','mean_router_confidence','mean_context_error','val_NDCG@10','val_HR@10','positions_per_s']
    display(h[[c for c in cols if c in h.columns]])
    if len(h):
        best=h.loc[h['val_NDCG@10'].idxmax()]
        print('BEST',best.to_dict())
        print('LC_V1_BEST_VAL_NDCG',0.040515991131704246)
        print('SASREC_VAL_NDCG',0.04296780165590764)
else:
    print('No history yet.')


## Crash recovery / final result


In [ ]:
rp=root/'result.json'; bp=root/'best.pt'; lp=root/'last.pt'
print('best.pt',bp.exists(),'last.pt',lp.exists(),'result.json',rp.exists())
if rp.exists():
    print(json.dumps(json.loads(rp.read_text()),indent=2))
elif lp.exists():
    ck=torch.load(lp,map_location='cpu')
    print('RECOVERABLE_FROM_EPOCH',ck['epoch'],'BEST_EPOCH',ck.get('best_epoch'),'BEST_VAL',ck.get('best'))
